# 🤖 Machine Learning — Notebook de Projeto

**Disciplina:** Machine Learning  
**Professor:** Messias Batista  
**Aluno(a):**  
**Data:**  
**Dataset:** German Credit Risk (credit.csv)  
**Problema de negócio:** Classificar o risco de crédito de clientes (bom ou mau pagador) com base em características pessoais e financeiras, utilizando Regressão Logística.  

---
## 1. 📦 Importações

Importe aqui todas as bibliotecas que serão utilizadas ao longo do projeto.

Você precisará de bibliotecas para:
- **Manipulação de dados** — leitura, transformação e análise de tabelas
- **Visualização** — criação de gráficos e figuras
- **Pré-processamento** — divisão dos dados, normalização e codificação de variáveis
- **Algoritmos de Machine Learning** — os modelos que serão treinados
- **Métricas de avaliação** — para medir o desempenho dos modelos

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

---
## 2. 📂 Carregamento dos Dados

Carregue o dataset e faça uma primeira inspeção para entender com o que você está trabalhando.

Nesta etapa você deve responder:
- Quantas linhas e colunas o dataset possui?
- Quais são os tipos de cada coluna?
- Existem valores nulos? Em quais colunas e em qual quantidade?

In [ ]:
credit_df = pd.read_csv('https://raw.githubusercontent.com/prof-mrafaelbatista/261_UNIESP_MBA-DADOS_machine_learning/refs/heads/main/datasets/credit.csv')
credit_df.head()

In [ ]:
print(f"Dimensões: {credit_df.shape}")
print(f"\nColunas: {list(credit_df.columns)}")
print(f"\nTipos de dados:\n{credit_df.dtypes}")
print(f"\nValores nulos por coluna:\n{credit_df.isnull().sum()}")

---
## 3. 🔍 Análise Exploratória de Dados (EDA)

Explore os dados antes de construir qualquer modelo. Esta é uma das etapas mais importantes do processo.

Nesta etapa você deve:
- Calcular estatísticas descritivas das variáveis numéricas
- Analisar a distribuição da variável alvo (está balanceada?)
- Visualizar a distribuição das demais variáveis
- Identificar possíveis outliers
- Analisar a correlação entre as variáveis

In [ ]:
# Estatísticas descritivas
credit_df.describe()

In [ ]:
# Distribuição da variável alvo
print(credit_df['Risk'].value_counts())

ax = credit_df['Risk'].value_counts().plot(
    kind='bar', figsize=(6, 4), title='Distribuição do Risco de Crédito',
    color=['steelblue', 'salmon']
)
ax.set_xlabel('Risco')
ax.set_ylabel('Quantidade')
ax.set_xticklabels(['Bom (good)', 'Mau (bad)'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição das variáveis numéricas por classe de risco
num_cols = ['Age', 'Credit amount', 'Duration']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(num_cols):
    for risk, grp in credit_df.groupby('Risk')[col]:
        axes[i].hist(grp, alpha=0.6, label=risk, bins=20)
    axes[i].set_title(col)
    axes[i].legend()
plt.suptitle('Distribuição por Variável e Risco')
plt.tight_layout()
plt.show()

# Contagem das variáveis categóricas
cat_cols = ['Job', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']
for col in cat_cols:
    print(f"\n{col}:\n{credit_df[col].value_counts()}")

---
## 4. 🛠️ Pré-processamento

Prepare os dados para que o modelo consiga aprender corretamente.

Nesta etapa você deve:
- Remover colunas que não contribuem para o modelo
- Tratar os valores nulos (remover ou preencher)
- Converter variáveis categóricas em numéricas
- Separar as features (X) da variável alvo (y)
- Aplicar normalização ou padronização se necessário

In [ ]:
# Criar variável alvo numérica: good=1, bad=0
credit_df['Risk_Num'] = np.where(credit_df['Risk'] == 'good', 1, 0)
print(credit_df['Risk_Num'].value_counts())

# Preencher NAs com categoria 'Null' (para não perder registros)
credit_df = credit_df.fillna('Null')

# Converter variáveis categóricas em dummies (one-hot encoding)
credit_df = pd.get_dummies(credit_df)

# Job é numérico — criar dummies manualmente
dummy_job = pd.get_dummies(credit_df['Job'])
dummy_job = dummy_job.rename(columns={0: 'Job 0', 1: 'Job 1', 2: 'Job 2', 3: 'Job 3'})
credit_df['Job_0'] = dummy_job['Job 0']
credit_df['Job_1'] = dummy_job['Job 1']
credit_df['Job_2'] = dummy_job['Job 2']
credit_df['Job_3'] = dummy_job['Job 3']

print(f"\nShape após pré-processamento: {credit_df.shape}")
credit_df.head()

In [ ]:
# Seleção das features (X) e variável alvo (y)
metrics = [
    'Age', 'Sex_female', 'Job_0', 'Job_1', 'Job_2',
    'Credit amount', 'Duration',
    'Housing_free', 'Housing_own',
    'Saving accounts_Null', 'Saving accounts_little',
    'Saving accounts_moderate', 'Saving accounts_quite rich',
    'Checking account_Null', 'Checking account_little', 'Checking account_moderate',
    'Purpose_business', 'Purpose_car', 'Purpose_domestic appliances',
    'Purpose_education', 'Purpose_furniture/equipment',
    'Purpose_radio/TV', 'Purpose_repairs',
]

x = credit_df[metrics]
y = credit_df['Risk_Num']

print(f"Features (X): {x.shape}")
print(f"Target  (y):  {y.shape}")
x.head()

---
## 5. ✂️ Separação Treino / Teste

Divida os dados em dois conjuntos: um para treinar o modelo e outro para testá-lo.

Lembre-se:
- O modelo deve ser treinado **apenas** com os dados de treino
- Os dados de teste simulam situações novas, que o modelo nunca viu
- Uma divisão comum é 70% treino e 30% teste
- Use o parâmetro `stratify` para manter a proporção das classes

In [ ]:
# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.30, stratify=y, random_state=42
)

# Padronização (StandardScaler): fit apenas no treino, transform em ambos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Treino: {X_train.shape}")
print(f"Teste:  {X_test.shape}")

---
## 6. 🧠 Treinamento do Modelo

Escolha um algoritmo, instancie o modelo e treine-o com os dados de treino.

Lembre-se:
- O treinamento acontece com o método `fit()`
- As previsões são feitas com o método `predict()`
- Você pode testar mais de um algoritmo e compará-los na seção 8

In [ ]:
# Instancia e treina a Regressão Logística
lr = LogisticRegression(random_state=42, max_iter=5000)
lr.fit(X_train_scaled, y_train.values.ravel())

In [ ]:
# Previsões para o conjunto de teste
y_pred = lr.predict(X_test_scaled)
y_pred

---
## 7. 📊 Avaliação do Modelo

Meça o desempenho do modelo utilizando métricas adequadas ao problema.

Para problemas de **classificação**, avalie:
- **Acurácia** — proporção de acertos em relação ao total
- **Matriz de Confusão** — visualização detalhada dos acertos e erros por classe
- **Relatório de Classificação** — precision, recall e f1-score por classe
- **Validação Cruzada** — para uma estimativa mais robusta e confiável do desempenho

In [ ]:
# Acurácia
print(f"Acurácia (Regressão Logística): {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# Matriz de Confusão
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Previsto Mau (0)', 'Previsto Bom (1)'],
            yticklabels=['Real Mau (0)', 'Real Bom (1)'])
plt.title('Matriz de Confusão — Regressão Logística')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

In [ ]:
# Relatório de Classificação
print(classification_report(y_test, y_pred, target_names=['Mau (0)', 'Bom (1)']))

In [ ]:
# Validação Cruzada (Pipeline para evitar data leakage no scaler)
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(random_state=42, max_iter=5000)),
])
scores = cross_val_score(pipeline_lr, x, y, scoring='accuracy', cv=5)
print(f"Scores por fold: {scores}")
print(f"Média: {scores.mean():.4f} ± {scores.std():.4f}")

---
## 8. 🏆 Comparação de Modelos

Teste outros algoritmos e compare os resultados para identificar o mais adequado ao seu problema.

Dica: use validação cruzada para comparar os modelos de forma justa,
pois ela elimina o efeito da aleatoriedade de uma única divisão treino/teste.

In [ ]:
# Comparação de modelos via cross-validation
modelos = {
    'Regressão Logística': Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(random_state=42, max_iter=5000)),
    ]),
    'Árvore de Decisão': DecisionTreeClassifier(random_state=42),
    'Random Forest':     RandomForestClassifier(n_estimators=100, random_state=42),
}

resultados = {}
for nome, modelo in modelos.items():
    scores = cross_val_score(modelo, x, y, scoring='accuracy', cv=5)
    resultados[nome] = scores
    print(f"{nome:<25} | Média: {scores.mean():.4f} ± {scores.std():.4f}")

# Boxplot comparativo
results_df = pd.DataFrame(resultados)
plt.figure(figsize=(9, 5))
sns.boxplot(data=results_df)
plt.title('Comparação de Modelos — Cross-Validation Accuracy')
plt.ylabel('Acurácia')
plt.tight_layout()
plt.show()

# -------------------------------------------------------
# Respondendo às perguntas de negócio
# -------------------------------------------------------
# Perfis dos clientes (23 features na ordem de `metrics`):
#   Age, Sex_female, Job_0..2, Credit amount, Duration,
#   Housing_free/own, Saving accounts_*, Checking account_*,
#   Purpose_*
#
# Cliente A — Gertrude Rocha: 60, female, Job 2, own, quite rich, NA, 2835, 24, furniture/equipment
# Cliente B — Abelardo Jurema: 53, male, Job 1, own, little, NA, 2835, 36, furniture/equipment
# Cliente C — Sérgio: 20, male, Job 2, free, moderate, moderate, 5866, 18, car

cliente1 = [60, 1, 0, 0, 1, 2835, 24, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0]
cliente2 = [53, 0, 0, 1, 0, 2835, 36, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0]
cliente3 = [20, 0, 0, 0, 1, 5866, 18, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0]

clientes_df = pd.DataFrame([cliente1, cliente2, cliente3], columns=metrics)
clientes_scaled = scaler.transform(clientes_df)
respostas = lr.predict(clientes_scaled)

nomes = ['Gertrude Rocha', 'Abelardo Jurema', 'Sérgio']
for nome, resp in zip(nomes, respostas):
    classificacao = 'BOM pagador ✅' if resp == 1 else 'MAU pagador ❌'
    print(f"{nome:<20} → {classificacao}")

---
## 9. 📝 Conclusões

Responda as perguntas abaixo com base nos resultados obtidos:

**1. Qual algoritmo apresentou melhor desempenho? Por quê?**  
> _Escreva aqui_

**2. O modelo está sofrendo overfitting ou underfitting? Como você identificou?**  
> _Escreva aqui_

**3. Os resultados respondem à pergunta de negócio levantada no início?**  
> _Escreva aqui_

**4. Qual seria o próximo passo para melhorar o modelo?**  
> _Escreva aqui_

---
### 🔖 Referências
- Dataset: German Credit Risk — https://github.com/prof-mrafaelbatista/261_UNIESP_MBA-DADOS_machine_learning
- Scikit-learn: https://scikit-learn.org
- Material da disciplina: www.mrafaelbatista.dev
